# 01 · Construcción del conjunto de datos

**Competencia:** Prueba Analítica: Modelo Opciones de Pago season 3 (Bancolombia, Kaggle).

**Objetivo del notebook:** construir el conjunto de datos de entrenamiento y de calificación para el modelo que pronostica, con un mes de anticipación, si una obligación en mora **aceptará una opción de pago**.

## Qué pide el enunciado

- Un modelo de pronóstico **para una ventana de un mes**: al cierre del mes `t` se califica lo que pasará en `t+1`.
- El archivo a calificar (`oot`) trae **solo llaves** de enero de 2024. No trae ninguna variable.
- Por lo tanto, todas las variables (X) deben construirse **con información disponible hasta el mes anterior** al de la etiqueta. Esto se conoce como *corte temporal en t-1* y es la regla central de este notebook.

## Regla de oro: nada del mes de la etiqueta entra en X

Para una fila cuya etiqueta es de agosto de 2023, las variables se calculan con datos hasta julio de 2023. Para las filas de enero de 2024 (test), con datos hasta diciembre de 2023. El train y el test se construyen con **exactamente el mismo procedimiento**, así el modelo se entrena en las mismas condiciones en las que se va a usar.

```
Mes:        ... 2023-07  2023-08  2023-09  2023-10  2023-11  2023-12 | 2024-01
Train:                      y        y        y        y        y    |
   X para y de 2023-08 ←── [historia hasta 2023-07]                  |
   X para y de 2023-12 ←────────────────── [historia hasta 2023-11]  |
Test:                                                                |    ?
   X para ? de 2024-01 ←──────────────────────── [historia hasta 2023-12]
```

## Fuentes de datos

La competencia entrega **5 fuentes** (más el `sample_submission`, que solo define el formato de entrega):

| # | Archivo | Grano | Periodo | Llave | Rol en el dataset |
|---|---|---|---|---|---|
| 1 | `..._trtest.csv` | obligación × mes | 2023-08 a 2023-12 | `nit`, `num_oblig_orig`, `num_oblig`, `fecha_var_rpta_alt` | **Etiqueta** `var_rpta_alt` y esqueleto del train. Sus otras columnas se usan solo **rezagadas**. |
| 2 | `..._oot.csv` | obligación | 2024-01 | mismas 4 llaves | Esqueleto del test. |
| 3 | `..._probabilidad_...csv` | obligación × mes | 2023-01 a 2023-12 | `num_oblig`, `fecha_corte` (AAAAMM) | Scores de los modelos internos del banco. |
| 4 | `..._maestra_cuotas_pagos_...csv` | obligación × mes | 2023-01 a 2023-12 | `num_oblig`, `fecha_corte` (AAAAMMDD) | Comportamiento de pago mensual. |
| 5 | `..._master_customer_data_...csv` | cliente × mes | 2023-07 a 2023-12 | `nit`, `year`, `month` | Demografía y finanzas del cliente. |

Las llaves que conectan todo:

- `nit_enmascarado` → identifica al **cliente**. Une con la fuente 5.
- `num_oblig_enmascarado` → identifica la **obligación**. Une con las fuentes 3 y 4.
- El **mes** → aparece con tres nombres y dos formatos distintos. Lo normalizamos a un índice entero `t` (2023-01 = 0, ..., 2023-12 = 11, 2024-01 = 12).

## Mapa de cruces

```
                 ┌──────────────────────────┐
                 │  ESQUELETO (train + test)│  una fila = (obligación, mes t)
                 │  llaves + var_rpta_alt   │
                 └────────────┬─────────────┘
                              │
      ┌───────────────┬───────┴────────┬────────────────┬──────────────────┐
      │               │                │                │                  │
 [3] prob         [4] pagos       [5] customer    [1] trtest rezagado   [1] trtest rezagado
 por num_oblig    por num_oblig   por nit         por num_oblig         por nit
 en t-1,t-2,t-3   ventanas 1/3/   último corte    lo que pasó con la    lo que pasó con el
                  6/12 meses      ≤ t-1           obligación antes de t cliente antes de t
```

## 0. Configuración

In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80, "display.width", 200, "display.max_rows", 120)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
OUT = DATA / "processed"
OUT.mkdir(parents=True, exist_ok=True)

F_TRAIN = DATA / "prueba_op_base_pivot_var_rpta_alt_enmascarado_trtest.csv"
F_OOT   = DATA / "prueba_op_base_pivot_var_rpta_alt_enmascarado_oot.csv"
F_PROB  = DATA / "prueba_op_probabilidad_oblig_base_hist_enmascarado_completa.csv"
F_PAGOS = DATA / "prueba_op_maestra_cuotas_pagos_mes_hist_enmascarado_completa.csv"
F_CLI   = DATA / "prueba_op_master_customer_data_enmascarado_completa.csv"

KEYS = ["nit_enmascarado", "num_oblig_orig_enmascarado", "num_oblig_enmascarado", "fecha_var_rpta_alt"]
N_MESES = 13  # índice 0 = 2023-01 ... 12 = 2024-01


def month_idx(yyyymm: pd.Series) -> pd.Series:
    "Convierte AAAAMM a un entero consecutivo: 202301 -> 0, 202312 -> 11, 202401 -> 12."
    return (yyyymm // 100 - 2023) * 12 + (yyyymm % 100 - 1)


def idx_to_yyyymm(i: int) -> int:
    return (2023 + i // 12) * 100 + (i % 12 + 1)


print("Raíz del proyecto:", ROOT)

Raíz del proyecto: C:\Users\USUARIO\OneDrive\Desktop\Bancolombia


## 1. Esqueleto: etiqueta (y) y llaves

El esqueleto es la lista de filas que el modelo va a aprender y a calificar. Cada fila es una **obligación en un mes `t`**.

- **Train**: las 568.251 filas de `trtest`, de las que tomamos solo las 4 llaves y la etiqueta `var_rpta_alt`.
- **Test**: las 112.549 filas de `oot`, que solo traen las 4 llaves. La etiqueta queda vacía.

### Por qué descartamos las otras 44 columnas de `trtest` en este paso

Esas columnas (`marca_alternativa`, `marca_alt_apli`, `cant_gestiones`, `pago_mes`, `dias_mora_fin`, ...) describen **lo que pasó durante el mismo mes de la etiqueta**: si aceptó, si se le aplicó la alternativa, cuántas llamadas recibió, cuánto pagó. En la exploración previa (`notebooks/02_eda.ipynb`) varias separan las clases con AUC de 0,83 a 0,97. Son el resultado, no un predictor. Además **no existen para enero de 2024**. Las recuperaremos más adelante, pero solo como historia de meses anteriores (sección 5).

In [2]:
tr_raw = pd.read_csv(F_TRAIN, low_memory=False)
oot_raw = pd.read_csv(F_OOT)

print("trtest:", tr_raw.shape, "| oot:", oot_raw.shape)
print("\nMeses en trtest:", sorted(tr_raw.fecha_var_rpta_alt.unique()))
print("Meses en oot:   ", sorted(oot_raw.fecha_var_rpta_alt.unique()))
print("\nDistribución de la etiqueta en trtest:")
print(tr_raw.var_rpta_alt.value_counts(normalize=True).rename("proporción").round(3))

trtest: (568251, 49) | oot: (112549, 4)

Meses en trtest: [np.int64(202308), np.int64(202309), np.int64(202310), np.int64(202311), np.int64(202312)]
Meses en oot:    [np.int64(202401)]

Distribución de la etiqueta en trtest:
var_rpta_alt
0    0.52
1    0.48
Name: proporción, dtype: float64


### Duplicados en el esqueleto

Con la llave de 3 campos (`nit`, `num_oblig`, mes) hay 238 filas repetidas en `trtest`. Al mirarlas, se trata de la **misma obligación con distinto `num_oblig_orig`** (obligaciones que cambiaron de número original, por ejemplo por una reestructuración). Como el `ID` de entrega incluye los tres identificadores, son filas distintas para la competencia y se conservan. Lo que sí verificamos es que la etiqueta coincida entre las copias, y que con la llave completa de 4 campos no haya repetidos.

In [3]:
K3 = ["nit_enmascarado", "num_oblig_enmascarado", "fecha_var_rpta_alt"]
dup3 = tr_raw[tr_raw.duplicated(K3, keep=False)]
print(f"Filas repetidas con llave de 3 campos (nit, num_oblig, mes): {len(dup3):,}")
print(f"  ... de las cuales tienen distinto num_oblig_orig: {dup3.groupby(K3).num_oblig_orig_enmascarado.nunique().gt(1).sum():,} grupos")
print(f"  ... con etiqueta distinta entre copias: {dup3.groupby(K3).var_rpta_alt.nunique().gt(1).sum()}")
print(f"Filas repetidas con la llave completa de 4 campos: {tr_raw.duplicated(KEYS).sum()}")

esq_train = tr_raw[KEYS + ["var_rpta_alt"]].drop_duplicates(KEYS).copy()
esq_train["conjunto"] = "train"

esq_test = oot_raw[KEYS].copy()
esq_test["var_rpta_alt"] = np.nan
esq_test["conjunto"] = "test"

base = pd.concat([esq_train, esq_test], ignore_index=True)
base["t"] = month_idx(base.fecha_var_rpta_alt)       # mes de la etiqueta
base["t_1"] = base["t"] - 1                           # último mes de información permitido
base["ID"] = (base.nit_enmascarado.astype(str) + "#"
              + base.num_oblig_orig_enmascarado.astype(str) + "#"
              + base.num_oblig_enmascarado.astype(str))

print(f"\nEsqueleto: {len(base):,} filas ({(base.conjunto=='train').sum():,} train + {(base.conjunto=='test').sum():,} test)")
base.groupby(["conjunto", "fecha_var_rpta_alt"]).size().rename("filas").to_frame()

Filas repetidas con llave de 3 campos (nit, num_oblig, mes): 475
  ... de las cuales tienen distinto num_oblig_orig: 237 grupos
  ... con etiqueta distinta entre copias: 0
Filas repetidas con la llave completa de 4 campos: 0



Esqueleto: 680,800 filas (568,251 train + 112,549 test)


filas
conjunto fecha_var_rpta_alt        
test     202401              112549
train    202308              113531
         202309              121185
         202310              115923
         202311              117146
         202312              100466

## 2. Fuente 3 · Probabilidades de los modelos internos

**Qué contiene:** para cada obligación y mes, tres scores que el banco ya calcula (`prob_propension` de pago, `prob_alrt_temprana` de entrar en mora, `prob_auto_cura` de ponerse al día solo) y el `lote` de priorización (1 = más prioritario).

**Cruce:** `num_oblig_enmascarado` + mes. Para la fila con etiqueta en `t` tomamos el registro del mes `t-1`. También traemos `t-2` y `t-3` para capturar tendencia (¿su propensión viene subiendo o bajando?).

**Decisiones:**
- Hay 3 llaves duplicadas; nos quedamos con la primera.
- Cobertura esperada: ~99,7 % de las filas tienen registro en `t-1` (ver `notebooks/02_eda.ipynb`).

In [4]:
prob = pd.read_csv(F_PROB)
prob["t"] = month_idx(prob.fecha_corte)
prob = prob.drop_duplicates(["num_oblig_enmascarado", "t"])
print("probabilidades:", prob.shape, "| meses:", prob.t.min(), "a", prob.t.max())
prob.head(3)

probabilidades: (4804833, 8) | meses: 0 a 11


,nit_enmascarado,num_oblig_enmascarado,fecha_corte,lote,prob_propension,prob_alrt_temprana,prob_auto_cura,t
0,296482,102381,202308,1,0.761350,0.193744,0.684784,7
1,391957,742315,202310,2,0.741803,0.384184,0.483696,9
2,229894,359919,202307,1,0.835373,0.285157,0.826225,6


In [5]:
PROB_COLS = ["prob_propension", "prob_alrt_temprana", "prob_auto_cura", "lote"]

feat_prob = base[["num_oblig_enmascarado", "t"]].copy()
for k in (1, 2, 3):
    lag = prob[["num_oblig_enmascarado", "t"] + PROB_COLS].copy()
    lag["t"] = lag["t"] + k                       # el registro del mes t-k se alinea con la fila de mes t
    lag = lag.rename(columns={c: f"prob_{c}_t{k}" for c in PROB_COLS})
    feat_prob = feat_prob.merge(lag, on=["num_oblig_enmascarado", "t"], how="left")

# Tendencias: promedio de 3 meses y cambio entre t-1 y t-3
for c in ["prob_propension", "prob_alrt_temprana", "prob_auto_cura"]:
    feat_prob[f"prob_{c}_mean3"] = feat_prob[[f"prob_{c}_t{k}" for k in (1, 2, 3)]].mean(axis=1)
    feat_prob[f"prob_{c}_delta13"] = feat_prob[f"prob_{c}_t1"] - feat_prob[f"prob_{c}_t3"]

feat_prob = feat_prob.drop(columns=["num_oblig_enmascarado", "t"])
print("Variables creadas:", feat_prob.shape[1])
print("Cobertura en t-1 por conjunto:")
print(feat_prob["prob_prob_propension_t1"].notna().groupby(base.conjunto).mean().round(3))
feat_prob.describe().T[["count", "mean", "min", "max"]].round(3)

Variables creadas: 18
Cobertura en t-1 por conjunto:
conjunto
test     0.998
train    0.997
Name: prob_prob_propension_t1, dtype: float64


,count,mean,min,max
prob_prob_propension_t1,678655.0,0.649,0.037,0.959
prob_prob_alrt_temprana_t1,678655.0,0.529,0.024,0.930
prob_prob_auto_cura_t1,678655.0,0.386,0.046,0.944
prob_lote_t1,678655.0,1.554,1.000,3.000
prob_prob_propension_t2,673759.0,0.706,0.037,0.959
prob_prob_alrt_temprana_t2,673759.0,0.469,0.024,0.930
prob_prob_auto_cura_t2,673759.0,0.452,0.046,0.945
prob_lote_t2,673759.0,1.538,1.000,3.000
prob_prob_propension_t3,658027.0,0.729,0.037,0.959
prob_prob_alrt_temprana_t3,658027.0,0.425,0.023,0.931


## 3. Fuente 4 · Historial de cuotas y pagos

**Qué contiene:** por obligación y mes, la cuota del mes, el pago total realizado, el porcentaje pagado, una marca de comportamiento (`PAGO_MAS`, `PAGO_MENOS`, `NO_PAGO`, `IGUAL`, `CANCELADO`, ...) y si hubo ajustes del banco (`REDIFERIDOS` señala que ya se le aplicó una opción de pago antes).

**Cruce:** `num_oblig_enmascarado` + mes, pero aquí no basta con el mes `t-1`: el comportamiento de pago es una **serie**. Construimos agregados sobre ventanas de 1, 3, 6 y 12 meses que terminan en `t-1`.

**Cómo se calcula eficientemente:** en vez de hacer 12 cruces, pivotamos cada variable a una matriz *obligación × mes* (13 columnas, una por mes) y calculamos las ventanas con sumas acumuladas. Cada fila del esqueleto indexa la matriz con su obligación y su `t`.

**Limpieza previa:**
- `fecha_corte` viene como AAAAMMDD (fin de mes). Se convierte a AAAAMM.
- `porc_pago` tiene valores infinitos (cuota en cero). Se reemplazan por nulo y se acota a 500 %.
- 28 llaves duplicadas: se conserva la primera.

In [6]:
pag = pd.read_csv(F_PAGOS, low_memory=False)
pag["t"] = month_idx(pag.fecha_corte // 100)
pag = pag.drop_duplicates(["num_oblig_enmascarado", "t"])
pag["porc_pago"] = pag.porc_pago.replace([np.inf, -np.inf], np.nan).clip(upper=500)

# Indicadores binarios que luego se cuentan por ventana
pag["pago_flag"]  = (pag.pago_total > 0).astype(float)
pag["no_pago"]    = (pag.marca_pago == "NO_PAGO").astype(float)
pag["pago_mas"]   = (pag.marca_pago == "PAGO_MAS").astype(float)
pag["cancelado"]  = (pag.marca_pago == "CANCELADO").astype(float)
pag["rediferido"] = (pag.ajustes_banco == "REDIFERIDOS").astype(float)
pag["ajuste"]     = (pag.ajustes_banco != "NO").astype(float)

print("cuotas y pagos:", pag.shape, "| meses:", pag.t.min(), "a", pag.t.max())
print("\nmarca_pago:"); print(pag.marca_pago.value_counts(normalize=True).round(3))

cuotas y pagos: (4855007, 20) | meses: 0 a 11

marca_pago:
marca_pago
PAGO_MAS                0.337
PAGO_MENOS              0.201
NO_PAGO                 0.169
IGUAL                   0.138
FACTURACION_MES_SGTE    0.117
CANCELADO               0.027
AJUSTES_BANCO           0.010
SIN_FACTURACION         0.000
Name: proportion, dtype: float64


### Matrices obligación × mes y agregados por ventana

Para cada variable numérica `v` construimos `M_v[i, m]` = valor de la obligación `i` en el mes `m` (nulo si no hay registro). Para una fila del esqueleto con obligación `i` y mes `t`, la ventana de `k` meses cubre las columnas `[t-k, t-1]`.

In [7]:
oblig_index = pd.Index(pag.num_oblig_enmascarado.unique())
row_pag = oblig_index.get_indexer(pag.num_oblig_enmascarado)


def to_wide(df, row_idx, col_t, values, n_rows, n_cols=N_MESES):
    M = np.full((n_rows, n_cols), np.nan)
    M[row_idx, col_t] = values
    return M


def window_sum_count(M, r, t, k):
    "Suma y conteo de valores no nulos de M[r, t-k : t] para cada par (r, t). r = -1 => nulo."
    V = np.nan_to_num(M, nan=0.0)
    C = (~np.isnan(M)).astype(float)
    cs = np.concatenate([np.zeros((M.shape[0], 1)), V.cumsum(axis=1)], axis=1)
    cc = np.concatenate([np.zeros((M.shape[0], 1)), C.cumsum(axis=1)], axis=1)
    lo = np.clip(t - k, 0, None)
    s = cs[r, t] - cs[r, lo]
    c = cc[r, t] - cc[r, lo]
    s[r < 0] = np.nan
    c[r < 0] = np.nan
    return s, c


def last_valid_before(M, r, t):
    "Último valor no nulo de M[r, :t] (propagación hacia adelante) y el mes en que ocurrió."
    df = pd.DataFrame(M)
    F = df.ffill(axis=1).to_numpy()
    idx = np.where(np.isnan(M), np.nan, np.arange(M.shape[1])[None, :])
    L = pd.DataFrame(idx).ffill(axis=1).to_numpy()
    val = F[r, t - 1]
    when = L[r, t - 1]
    val[r < 0] = np.nan
    when[r < 0] = np.nan
    return val, when


r_pag = oblig_index.get_indexer(base.num_oblig_enmascarado)
t_arr = base.t.to_numpy()

mats = {v: to_wide(pag, row_pag, pag.t.to_numpy(), pag[v].to_numpy(), len(oblig_index))
        for v in ["valor_cuota_mes", "pago_total", "porc_pago", "pago_flag", "no_pago", "pago_mas", "cancelado", "rediferido", "ajuste"]}

feat_pag = pd.DataFrame(index=base.index)

# Historial total disponible y recencia
_, n_hist = window_sum_count(mats["pago_flag"], r_pag, t_arr, 12)
feat_pag["pag_meses_historial"] = n_hist
_, ult_pago_when = last_valid_before(np.where(mats["pago_flag"] == 1, 1.0, np.nan), r_pag, t_arr)
feat_pag["pag_meses_desde_ultimo_pago"] = t_arr - ult_pago_when

# Ventanas
for k in (1, 3, 6, 12):
    s, c = window_sum_count(mats["pago_total"], r_pag, t_arr, k)
    feat_pag[f"pag_pago_total_sum_{k}m"] = s
    feat_pag[f"pag_pago_total_mean_{k}m"] = s / c.clip(min=1)
    s, c = window_sum_count(mats["valor_cuota_mes"], r_pag, t_arr, k)
    feat_pag[f"pag_cuota_mean_{k}m"] = s / c.clip(min=1)
    s, c = window_sum_count(mats["porc_pago"], r_pag, t_arr, k)
    feat_pag[f"pag_porc_pago_mean_{k}m"] = s / c.clip(min=1)
    for v in ["pago_flag", "no_pago", "pago_mas", "rediferido", "ajuste", "cancelado"]:
        s, c = window_sum_count(mats[v], r_pag, t_arr, k)
        feat_pag[f"pag_n_{v}_{k}m"] = s

# Razones útiles: pago sobre cuota y tendencia de la cuota
feat_pag["pag_ratio_pago_cuota_3m"] = feat_pag["pag_pago_total_sum_3m"] / feat_pag["pag_cuota_mean_3m"].replace(0, np.nan) / 3
feat_pag["pag_cuota_trend_1_vs_6"] = feat_pag["pag_cuota_mean_1m"] / feat_pag["pag_cuota_mean_6m"].replace(0, np.nan)

# Estado en t-1: marca de pago y descriptores de la obligación (producto, aplicativo, segmento)
estado_t1 = pag[["num_oblig_enmascarado", "t", "marca_pago", "ajustes_banco", "producto", "aplicativo", "segmento"]].copy()
estado_t1["t"] = estado_t1["t"] + 1
estado_t1 = estado_t1.rename(columns={"marca_pago": "pag_marca_pago_t1", "ajustes_banco": "pag_ajustes_banco_t1",
                                      "producto": "obl_producto", "aplicativo": "obl_aplicativo", "segmento": "obl_segmento"})
feat_pag = feat_pag.join(base[["num_oblig_enmascarado", "t"]].merge(estado_t1, on=["num_oblig_enmascarado", "t"], how="left")
                         .drop(columns=["num_oblig_enmascarado", "t"]))

print("Variables creadas:", feat_pag.shape[1])
print("Cobertura (algún registro en los 12 meses previos):")
print(feat_pag.pag_meses_historial.gt(0).groupby(base.conjunto).mean().round(3))
feat_pag[["pag_meses_historial", "pag_meses_desde_ultimo_pago", "pag_n_pago_flag_3m", "pag_n_no_pago_6m",
          "pag_porc_pago_mean_3m", "pag_n_rediferido_12m", "pag_marca_pago_t1", "obl_producto"]].head()

Variables creadas: 49
Cobertura (algún registro en los 12 meses previos):
conjunto
test     0.998
train    0.998
Name: pag_meses_historial, dtype: float64


,pag_meses_historial,pag_meses_desde_ultimo_pago,pag_n_pago_flag_3m,pag_n_no_pago_6m,pag_porc_pago_mean_3m,pag_n_rediferido_12m,pag_marca_pago_t1,obl_producto
0,7.0,1.0,2.0,3.0,109.000000,1.0,PAGO_MENOS,TARJETA DE CREDITO
1,11.0,1.0,3.0,0.0,333.666667,0.0,PAGO_MENOS,LIBRE INVERSION
2,11.0,2.0,2.0,1.0,228.000000,0.0,NO_PAGO,LIBRE INVERSION
3,4.0,2.0,2.0,1.0,167.000000,0.0,NO_PAGO,ROTATIVOS
4,10.0,1.0,3.0,0.0,167.333333,0.0,PAGO_MENOS,ROTATIVOS


### Agregados a nivel de cliente desde pagos

Un cliente puede tener varias obligaciones. Cuántas tiene, cuánta cuota suma y en cuántas dejó de pagar el mes anterior describe su carga financiera total, algo que la obligación sola no muestra.

In [8]:
cli_pag = (pag.groupby(["nit_enmascarado", "t"])
              .agg(pagcli_n_oblig_t1=("num_oblig_enmascarado", "nunique"),
                   pagcli_cuota_total_t1=("valor_cuota_mes", "sum"),
                   pagcli_pago_total_t1=("pago_total", "sum"),
                   pagcli_n_no_pago_t1=("no_pago", "sum"),
                   pagcli_n_rediferido_t1=("rediferido", "sum"))
              .reset_index())
cli_pag["t"] = cli_pag["t"] + 1
feat_pagcli = (base[["nit_enmascarado", "t"]].merge(cli_pag, on=["nit_enmascarado", "t"], how="left")
               .drop(columns=["nit_enmascarado", "t"]))
feat_pagcli["pagcli_ratio_pago_cuota_t1"] = feat_pagcli.pagcli_pago_total_t1 / feat_pagcli.pagcli_cuota_total_t1.replace(0, np.nan)
print("Variables creadas:", feat_pagcli.shape[1])
feat_pagcli.describe().T[["count", "mean", "50%", "max"]].round(2)

Variables creadas: 6


,count,mean,50%,max
pagcli_n_oblig_t1,680179.0,3.42,2.00,6.700000e+01
pagcli_cuota_total_t1,680179.0,2053710.87,838739.00,5.071144e+08
pagcli_pago_total_t1,680179.0,2600783.03,212247.00,1.667793e+09
pagcli_n_no_pago_t1,680179.0,1.37,0.00,5.900000e+01
pagcli_n_rediferido_t1,680179.0,0.02,0.00,5.000000e+00
pagcli_ratio_pago_cuota_t1,674117.0,1.70,0.33,2.585271e+04


## 4. Fuente 5 · Master customer (demografía y finanzas del cliente)

**Qué contiene:** características del cliente (edad, género, estado civil, ocupación, nivel académico, ingresos, activos, pasivos, patrimonio, segmento comercial, región, antigüedad) en 6 cortes mensuales, julio a diciembre de 2023. Cada cliente aparece en promedio en 1,8 cortes, no en los 6.

**Cruce:** por `nit_enmascarado`. Como no hay un corte por mes para cada cliente, se toma el **último corte disponible con fecha ≤ t-1** (`merge_asof` hacia atrás).

**Decisión documentada:** la tabla empieza en julio de 2023, así que para las etiquetas de agosto solo el 24 % de los clientes tiene un corte anterior. Como estas variables cambian lentamente y no dependen del resultado de la gestión, cuando no existe corte anterior se usa el **primer corte posterior** y se marca con `cli_corte_posterior = 1`, para que el modelo y el lector sepan que ese dato se tomó fuera de la ventana estricta. Con eso la cobertura sube a ~80 % en todos los meses, igual que en el test.

**Limpieza:** ingresos, activos y pasivos tienen máximos absurdos (10^12). Se aplica `log1p` para comprimir la escala. Edad 0 o mayor a 100 se pasa a nulo.

In [9]:
cli = pd.read_csv(F_CLI, low_memory=False)
cli["t"] = month_idx(cli.year * 100 + cli.month)
cli = cli.sort_values(["nit_enmascarado", "t"]).drop_duplicates(["nit_enmascarado", "t"], keep="last")

CLI_NUM = ["edad_cli", "num_hijos", "personas_dependientes", "total_ing", "tot_activos", "tot_pasivos",
           "egresos_mes", "tot_patrimonio"]
CLI_CAT = ["tipo_cli", "genero_cli", "estado_civil", "tipo_vivienda", "nivel_academico", "ocup",
           "declarante", "segm", "subsegm", "region_of", "cli_actualizado", "nicho"]

for c in CLI_NUM:
    cli[c] = pd.to_numeric(cli[c], errors="coerce")
cli.loc[(cli.edad_cli <= 0) | (cli.edad_cli > 100), "edad_cli"] = np.nan
for c in ["total_ing", "tot_activos", "tot_pasivos", "egresos_mes", "tot_patrimonio"]:
    cli[c] = np.log1p(cli[c].clip(lower=0))
cli["antiguedad_meses"] = ((cli.year * 12 + cli.month)
                           - (pd.to_numeric(cli.f_vinc, errors="coerce") // 10000 * 12
                              + pd.to_numeric(cli.f_vinc, errors="coerce") // 100 % 100))
cli["antiguedad_meses"] = cli.antiguedad_meses.where(cli.antiguedad_meses.between(0, 900))
cli["ratio_pasivo_activo"] = np.expm1(cli.tot_pasivos) / np.expm1(cli.tot_activos).replace(0, np.nan)
cli["ratio_egreso_ingreso"] = np.expm1(cli.egresos_mes) / np.expm1(cli.total_ing).replace(0, np.nan)

CLI_FEATS = CLI_NUM + CLI_CAT + ["antiguedad_meses", "ratio_pasivo_activo", "ratio_egreso_ingreso"]
cli_small = cli[["nit_enmascarado", "t"] + CLI_FEATS].rename(columns={c: f"cli_{c}" for c in CLI_FEATS})
cli_small = cli_small.rename(columns={"t": "cli_t_corte"}).sort_values("cli_t_corte")

print("master customer:", cli.shape, "| clientes únicos:", cli.nit_enmascarado.nunique(), "| cortes:", sorted(cli.t.unique()))

master customer: (429999, 41) | clientes únicos: 241049 | cortes: [np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]


In [10]:
q = base[["nit_enmascarado", "t_1"]].reset_index().sort_values("t_1")

# 1) último corte <= t-1
back = pd.merge_asof(q, cli_small, left_on="t_1", right_on="cli_t_corte", by="nit_enmascarado", direction="backward")
# 2) respaldo: primer corte > t-1, solo para quienes no tuvieron el anterior
fwd = pd.merge_asof(q, cli_small, left_on="t_1", right_on="cli_t_corte", by="nit_enmascarado", direction="forward")

sin_back = back.cli_t_corte.isna()
feat_cli = back.copy()
feat_cli.loc[sin_back, fwd.columns] = fwd.loc[sin_back].values
feat_cli["cli_corte_posterior"] = (sin_back & feat_cli.cli_t_corte.notna()).astype(int)
feat_cli = feat_cli.set_index("index").sort_index().drop(columns=["nit_enmascarado", "t_1"])

print("Variables creadas:", feat_cli.shape[1])
print("\nCobertura por mes de etiqueta (con corte anterior / con cualquier corte):")
cob = pd.DataFrame({"corte_anterior": (~sin_back).groupby(base.loc[back["index"].values].t.values).mean().round(3),
                    "cualquier_corte": feat_cli.cli_t_corte.notna().groupby(base.t).mean().round(3)})
cob.index = [idx_to_yyyymm(i) for i in cob.index]
cob

Variables creadas: 25

Cobertura por mes de etiqueta (con corte anterior / con cualquier corte):


,corte_anterior,cualquier_corte
202308,0.237,0.801
202309,0.419,0.804
202310,0.559,0.803
202311,0.662,0.805
202312,0.741,0.802
202401,0.804,0.804


## 5. Fuente 1 rezagada · Lo que ya sabemos de la obligación y del cliente por meses anteriores

Aquí recuperamos las columnas de `trtest` que descartamos en el paso 1, pero **solo de meses anteriores a `t`**. Así son legítimas: describen lo que pasó con esa obligación en la gestión de meses pasados.

Esto es especialmente valioso porque la etiqueta tiene memoria: en la exploración vimos que una obligación que aceptó el mes anterior vuelve a aceptar el 65 % de las veces.

**Cobertura limitada, y por qué es importante:** solo el 22 % de las filas del train tiene a la misma obligación en `t-1`, y el 49 % de las obligaciones del test aparecen en algún mes del train. El resto son obligaciones que entran en mora por primera vez en la ventana. Para ellas estas variables quedan nulas, y eso mismo es información: `lag_n_meses_vistos = 0` significa "cliente sin historial de gestión reciente".

**Efecto del arranque de la ventana:** `trtest` empieza en agosto de 2023, así que las filas de agosto no pueden tener historial rezagado (todas quedan con `lag_n_meses_vistos = 0`), septiembre tiene hasta un mes, y así sucesivamente. El test de enero tiene hasta cinco meses. Esto se refleja en la tabla de la sección 7 y es una consideración para el modelado: la distribución de estas variables cambia con el mes de la etiqueta.

Dos niveles:
- **Obligación**: ¿aceptó antes? ¿cuántas veces? ¿cuántas opciones tenía preaprobadas? ¿cuánta mora, saldo y deuda tenía la última vez? ¿cuántas gestiones recibió?
- **Cliente**: ¿cuántas obligaciones suyas fueron gestionadas antes y en cuántas aceptó?

In [11]:
tr = tr_raw.drop_duplicates(KEYS).copy()
tr["t"] = month_idx(tr.fecha_var_rpta_alt)

LAG_NUM = ["var_rpta_alt", "cant_alter_posibles", "cant_gestiones", "rpc", "promesas_cumplidas", "cant_acuerdo",
           "min_mora", "max_mora", "dias_mora_fin", "vlr_obligacion", "vlr_vencido", "saldo_capital", "endeudamiento",
           "valor_cuota_mes", "pago_mes", "porc_pago_cuota"]
LAG_CAT = ["producto", "producto_cons", "banca", "segmento", "aplicativo", "rango_mora", "desc_alternativa1",
           "alter_posible1_2", "marca_alt_rank", "alternativa_aplicada_agr", "marca_pago"]

oblig_tr_index = pd.Index(tr.num_oblig_enmascarado.unique())
row_tr = oblig_tr_index.get_indexer(tr.num_oblig_enmascarado)
r_tr = oblig_tr_index.get_indexer(base.num_oblig_enmascarado)

feat_lag = pd.DataFrame(index=base.index)

# --- Obligación: numéricas (último valor conocido antes de t, y agregados de aceptación)
for v in LAG_NUM:
    M = to_wide(tr, row_tr, tr.t.to_numpy(), pd.to_numeric(tr[v], errors="coerce").to_numpy(), len(oblig_tr_index))
    val, when = last_valid_before(M, r_tr, t_arr)
    feat_lag[f"lag_{v}_ult"] = val
    if v == "var_rpta_alt":
        feat_lag["lag_meses_desde_ultima_gestion"] = t_arr - when
        s, c = window_sum_count(M, r_tr, t_arr, 12)
        feat_lag["lag_n_meses_vistos"] = np.nan_to_num(c, nan=0.0)
        feat_lag["lag_n_aceptos"] = np.nan_to_num(s, nan=0.0)
        feat_lag["lag_tasa_acepto"] = np.where(c > 0, s / np.where(c > 0, c, 1), np.nan)
        feat_lag["lag_acepto_t1"] = M[r_tr, t_arr - 1]
        feat_lag.loc[r_tr < 0, "lag_acepto_t1"] = np.nan

# --- Obligación: categóricas (último valor conocido antes de t), vía códigos enteros
for v in LAG_CAT:
    cat = tr[v].astype("category")
    M = to_wide(tr, row_tr, tr.t.to_numpy(), cat.cat.codes.astype(float).to_numpy(), len(oblig_tr_index))
    val, _ = last_valid_before(M, r_tr, t_arr)
    codes = np.where(np.isnan(val), -1, val).astype(int)
    feat_lag[f"lag_{v}_ult"] = pd.Categorical.from_codes(codes, categories=cat.cat.categories)

# --- Cliente: obligaciones gestionadas y aceptadas en meses anteriores
nit_index = pd.Index(tr.nit_enmascarado.unique())
cli_m = tr.groupby(["nit_enmascarado", "t"]).agg(n=("num_oblig_enmascarado", "nunique"), a=("var_rpta_alt", "sum")).reset_index()
rn = nit_index.get_indexer(cli_m.nit_enmascarado)
Mn = to_wide(cli_m, rn, cli_m.t.to_numpy(), cli_m.n.to_numpy().astype(float), len(nit_index))
Ma = to_wide(cli_m, rn, cli_m.t.to_numpy(), cli_m.a.to_numpy().astype(float), len(nit_index))
r_nit = nit_index.get_indexer(base.nit_enmascarado)
sn, _ = window_sum_count(Mn, r_nit, t_arr, 12)
sa, _ = window_sum_count(Ma, r_nit, t_arr, 12)
feat_lag["lagcli_n_oblig_gestionadas"] = np.nan_to_num(sn, nan=0.0)
feat_lag["lagcli_n_aceptos"] = np.nan_to_num(sa, nan=0.0)
feat_lag["lagcli_tasa_acepto"] = np.where(sn > 0, sa / np.where(sn > 0, sn, 1), np.nan)

print("Variables creadas:", feat_lag.shape[1])
print("\nCobertura del historial de la obligación (lag_n_meses_vistos > 0):")
print(feat_lag.lag_n_meses_vistos.gt(0).groupby(base.conjunto).mean().round(3))
print("\nSeñal: tasa de aceptación en t según si aceptó la última vez que se gestionó (solo train):")
m = base.conjunto.eq("train")
print(base[m].groupby(feat_lag.loc[m, "lag_var_rpta_alt_ult"].fillna(-1).map({-1: "sin historial", 0: "no aceptó", 1: "aceptó"}))
      .var_rpta_alt.agg(filas="size", tasa_acepto="mean").round(3))

Variables creadas: 35

Cobertura del historial de la obligación (lag_n_meses_vistos > 0):
conjunto
test     0.486
train    0.295
Name: lag_n_meses_vistos, dtype: float64

Señal: tasa de aceptación en t según si aceptó la última vez que se gestionó (solo train):
                       filas  tasa_acepto
lag_var_rpta_alt_ult                     
aceptó                 61545        0.648
no aceptó             105965        0.348
sin historial         400741        0.489


## 6. Ensamble final

Unimos el esqueleto con los cinco bloques de variables. Todo se hace por posición (mismo índice del esqueleto), así que no hay riesgo de duplicar filas en el cruce.

Prefijos de las variables para saber de qué fuente salen:

| Prefijo | Fuente | Descripción |
|---|---|---|
| `prob_` | probabilidades | Scores internos en t-1, t-2, t-3 y tendencias |
| `pag_` | cuotas y pagos | Comportamiento de pago de la obligación en ventanas de 1/3/6/12 meses hasta t-1 |
| `obl_` | cuotas y pagos | Producto, aplicativo y segmento de la obligación en t-1 |
| `pagcli_` | cuotas y pagos | Carga financiera total del cliente en t-1 |
| `cli_` | master customer | Demografía y finanzas del cliente, último corte disponible |
| `lag_` | trtest rezagado | Historia de gestión y aceptación de la obligación antes de t |
| `lagcli_` | trtest rezagado | Historia de aceptación del cliente antes de t |

In [12]:
dataset = pd.concat([base, feat_prob, feat_pag, feat_pagcli, feat_cli, feat_lag], axis=1)

ID_COLS = KEYS + ["ID", "conjunto", "t", "t_1"]
TARGET = "var_rpta_alt"
FEATURES = [c for c in dataset.columns if c not in ID_COLS + [TARGET]]

print(f"Dataset: {dataset.shape[0]:,} filas x {dataset.shape[1]} columnas")
print(f"Variables (X): {len(FEATURES)} | Etiqueta (y): {TARGET} | Identificadores: {len(ID_COLS)}")
print("\nVariables por fuente:")
print(pd.Series([c.split('_')[0] for c in FEATURES]).value_counts().rename("n_variables").to_frame())

Dataset: 680,800 filas x 142 columnas
Variables (X): 133 | Etiqueta (y): var_rpta_alt | Identificadores: 8

Variables por fuente:
        n_variables
pag              46
lag              32
cli              25
prob             18
pagcli            6
obl               3
lagcli            3


## 7. Verificación de fuga temporal (leakage)

Tres comprobaciones que dan confianza de que ninguna variable usa información del mes de la etiqueta o posterior:

1. **Por construcción**: los cruces con probabilidades y pagos se hicieron desplazando el mes de la fuente (`t + k`), y las ventanas terminan en `t-1`. El `merge_asof` de demografía usa `t-1` como límite, y cuando se tomó un corte posterior quedó marcado.
2. **Por datos**: el corte de demografía usado nunca supera `t-1` salvo en las filas marcadas con `cli_corte_posterior`.
3. **Por el test**: para enero de 2024 ninguna fuente tiene datos de enero, así que si el test tiene valores, salieron de diciembre o antes. Comprobamos que la distribución de las variables en el test se parezca a la de diciembre de 2023 (si el procedimiento es el mismo, deben parecerse).

In [13]:
# 2) cortes de demografía
viol = dataset[(dataset.cli_t_corte > dataset.t_1) & (dataset.cli_corte_posterior == 0)]
print("Filas con corte de demografía posterior a t-1 sin marcar:", len(viol))

# 3) test vs último mes del train: medias de algunas variables clave
chk = ["prob_prob_propension_t1", "pag_n_pago_flag_3m", "pag_porc_pago_mean_3m", "pag_meses_historial",
       "cli_edad_cli", "lag_n_meses_vistos", "lag_tasa_acepto", "pagcli_n_oblig_t1"]
comp = dataset.groupby(dataset.fecha_var_rpta_alt)[chk].mean().round(3).T
comp.columns = [f"{c}" for c in comp.columns]
comp

Filas con corte de demografía posterior a t-1 sin marcar: 0


,202308,202309,202310,202311,202312,202401
prob_prob_propension_t1,0.658,0.659,0.656,0.658,0.627,0.634
pag_n_pago_flag_3m,1.911,1.920,1.869,1.834,1.719,1.770
pag_porc_pago_mean_3m,142.504,144.762,140.547,139.330,127.075,138.390
pag_meses_historial,6.421,7.279,8.061,8.831,9.552,10.294
cli_edad_cli,40.768,40.872,41.012,40.964,40.758,40.914
lag_n_meses_vistos,0.000,0.257,0.437,0.543,0.695,0.759
lag_tasa_acepto,NaN,0.340,0.375,0.358,0.386,0.417
pagcli_n_oblig_t1,3.405,3.342,3.415,3.590,3.472,3.276


## 8. Cobertura y nulos por bloque

Los nulos no se imputan aquí: se dejan para el notebook de modelado, donde la decisión depende del algoritmo (los modelos de árboles como LightGBM los manejan nativamente). Lo que sí registramos es **de dónde vienen** los nulos, porque tienen significado de negocio: obligación nueva sin historial, cliente sin ficha demográfica, etc.

In [14]:
nulos = (dataset[FEATURES].isna().groupby(dataset.conjunto).mean().T * 100).round(1)
nulos["bloque"] = [c.split("_")[0] for c in nulos.index]
resumen_nulos = nulos.groupby("bloque")[["train", "test"]].agg(["min", "median", "max"]).round(1)
resumen_nulos.columns = ["_".join(c) for c in resumen_nulos.columns]
resumen_nulos

,train_min,train_median,train_max,test_min,test_median,test_max
bloque,,,,,,
cli,0.0,19.8,73.5,0.0,19.7,73.2
lag,0.0,70.5,95.2,0.0,51.4,87.4
lagcli,0.0,0.0,62.5,0.0,0.0,37.3
obl,0.2,0.2,0.2,0.2,0.2,0.2
pag,0.0,0.0,5.5,0.2,0.2,5.5
pagcli,0.1,0.1,0.9,0.1,0.1,1.4
prob,0.3,1.0,3.3,0.2,1.0,3.8


In [15]:
print("Variables con más del 50 % de nulos en el test:")
nulos[nulos.test > 50].sort_values("test", ascending=False)

Variables con más del 50 % de nulos en el test:


conjunto,test,train,bloque
lag_alternativa_aplicada_agr_ult,87.4,95.2,lag
lag_acepto_t1,74.2,78.1,lag
cli_tipo_vivienda,73.2,73.5,cli
cli_nivel_academico,63.8,63.4,cli
cli_nicho,61.5,62.0,cli
lag_cant_acuerdo_ult,52.5,71.2,lag
lag_cant_gestiones_ult,52.5,71.2,lag
lag_tasa_acepto,51.4,70.5,lag
lag_meses_desde_ultima_gestion,51.4,70.5,lag
lag_cant_alter_posibles_ult,51.4,70.5,lag


## 9. Guardado

Se guardan dos archivos en formato Parquet (conserva tipos y pesa mucho menos que CSV):

- `data/processed/train.parquet` → filas de 2023-08 a 2023-12 con etiqueta.
- `data/processed/test.parquet` → filas de 2024-01 sin etiqueta, con la columna `ID` lista para el archivo de entrega.

Además, `docs/diccionario_dataset.md` con la lista de variables, su fuente y su tipo, para el documento técnico.

In [16]:
for c in dataset.columns:
    if dataset[c].dtype == object:
        dataset[c] = dataset[c].astype("category")

train_df = dataset[dataset.conjunto == "train"].drop(columns=["conjunto"]).reset_index(drop=True)
test_df = dataset[dataset.conjunto == "test"].drop(columns=["conjunto", TARGET]).reset_index(drop=True)

train_df.to_parquet(OUT / "train.parquet", index=False)
test_df.to_parquet(OUT / "test.parquet", index=False)
pd.Series(FEATURES, name="feature").to_csv(OUT / "features.csv", index=False)

print("train:", train_df.shape, "->", OUT / "train.parquet", f"({(OUT/'train.parquet').stat().st_size/1e6:.1f} MB)")
print("test: ", test_df.shape, "->", OUT / "test.parquet", f"({(OUT/'test.parquet').stat().st_size/1e6:.1f} MB)")

train: (568251, 141) -> C:\Users\USUARIO\OneDrive\Desktop\Bancolombia\data\processed\train.parquet (180.0 MB)
test:  (112549, 140) -> C:\Users\USUARIO\OneDrive\Desktop\Bancolombia\data\processed\test.parquet (47.4 MB)


In [17]:
DESC_BLOQUE = {
    "prob": "Scores de modelos internos del banco (fuente: probabilidades), mes t-1, t-2, t-3",
    "pag": "Comportamiento de pago de la obligación en ventanas hasta t-1 (fuente: cuotas y pagos)",
    "obl": "Descriptores de la obligación en t-1 (fuente: cuotas y pagos)",
    "pagcli": "Carga financiera total del cliente en t-1 (fuente: cuotas y pagos)",
    "cli": "Demografía y finanzas del cliente, último corte <= t-1 (fuente: master customer)",
    "lag": "Historia de gestión y aceptación de la obligación antes de t (fuente: trtest rezagado)",
    "lagcli": "Historia de aceptación del cliente antes de t (fuente: trtest rezagado)",
}
rows = []
for c in FEATURES:
    b = c.split("_")[0]
    rows.append({"variable": c, "bloque": b, "tipo": str(dataset[c].dtype),
                 "nulos_train_%": nulos.loc[c, "train"], "nulos_test_%": nulos.loc[c, "test"]})
dic = pd.DataFrame(rows)

lines = ["# Diccionario del dataset procesado", "",
         "Generado por `notebooks/01_construccion_dataset.ipynb`.", "",
         f"- Filas train: {len(train_df):,} (2023-08 a 2023-12) | Filas test: {len(test_df):,} (2024-01)",
         f"- Etiqueta: `{TARGET}` | Identificadores: {', '.join('`'+c+'`' for c in ID_COLS)}",
         f"- Variables: {len(FEATURES)}", "",
         "## Bloques", ""]
for b, d in DESC_BLOQUE.items():
    lines.append(f"- **{b}_**: {d} ({(dic.bloque == b).sum()} variables)")
lines += ["", "## Variables", "", dic.to_markdown(index=False)]
(ROOT / "docs" / "diccionario_dataset.md").write_text("\n".join(lines), encoding="utf-8")
print("Diccionario escrito en docs/diccionario_dataset.md")
dic.groupby("bloque").size().rename("n_variables").to_frame()

Diccionario escrito en docs/diccionario_dataset.md


,n_variables
bloque,
cli,25
lag,32
lagcli,3
obl,3
pag,46
pagcli,6
prob,18


## 10. Resumen

- **5 fuentes**, unidas sobre un esqueleto de (obligación, mes) con **corte estricto en t-1**. Train y test se construyen con el mismo código.
- **Etiqueta**: `var_rpta_alt` de `trtest`. **Test**: 112.549 obligaciones de enero de 2024.
- **Bloques de variables**: scores internos (prob), pagos en ventanas (pag, pagcli), descriptores de la obligación (obl), demografía (cli) e historia de gestión rezagada (lag, lagcli).
- **Decisiones explícitas**: se descartan las columnas de `trtest` del mismo mes por fuga; los duplicados se eliminan porque son copias exactas; demografía sin corte anterior se toma del primer corte posterior y se marca; valores infinitos y extremos se acotan o se pasan a logaritmo; los nulos se conservan porque llevan información.
- **Siguiente paso** (`02_modelado.ipynb`): validación temporal entrenando con 2023-08 a 2023-11 y midiendo F1 en 2023-12, selección de umbral, y entrenamiento final con los 5 meses para calificar enero.